In [3]:
# Install dependencies (run this once)
# pip install transformers torch
import torch
from transformers import pipeline

# Load a pre-trained sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis")

# Test it
texts = [
    "I love this product! It's amazing ❤️",
    "This is the worst experience I’ve ever had.",
    "The movie was okay, not too great but not terrible either."
]

for text in texts:
    result = sentiment_analyzer(text)[0]
    print(f"Text: {text}")
    print(f"Label: {result['label']}, Score: {result['score']:.4f}")
    print("----")


ModuleNotFoundError: No module named 'torch'

In [4]:

pip install https://github.com/jllllll/bitsandbytes-windows-webui/releases/download/wheels/bitsandbytes-0.41.0-py3-none-win_amd64.whl

     ---------------------------------------- 0.0/152.7 MB ? eta -:--:--
     ---------------------------------------- 0.3/152.7 MB ? eta -:--:--
      --------------------------------------- 2.4/152.7 MB 8.9 MB/s eta 0:00:17
     - -------------------------------------- 3.9/152.7 MB 8.2 MB/s eta 0:00:19
     - -------------------------------------- 5.2/152.7 MB 7.8 MB/s eta 0:00:19
     - -------------------------------------- 6.8/152.7 MB 7.3 MB/s eta 0:00:20
     -- ------------------------------------- 8.4/152.7 MB 7.4 MB/s eta 0:00:20
     -- ------------------------------------- 9.7/152.7 MB 7.2 MB/s eta 0:00:20
     -- ------------------------------------ 10.7/152.7 MB 6.9 MB/s eta 0:00:21
     --- ----------------------------------- 12.1/152.7 MB 6.8 MB/s eta 0:00:21
     --- ----------------------------------- 13.4/152.7 MB 6.7 MB/s eta 0:00:21
     --- ----------------------------------- 14.9/152.7 MB 6.8 MB/s eta 0:00:21
     ---- ---------------------------------- 16.5/152.


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# ---- STEP 1: Load LLM ----
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ---- STEP 2: Load Embedding Model ----
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# ---- STEP 3: Ingest Company Docs ----
documents = [
    "Our company was founded in 2015 and specializes in renewable energy solutions.",
    "We provide solar panels, wind turbines, and energy storage systems.",
    "Customer support is available 24/7 via email and phone.",
    "Headquarters are located in New Delhi, India."
]

# Create embeddings
doc_embeddings = embedder.encode(documents)
dimension = doc_embeddings.shape[1]

# ---- STEP 4: Build FAISS Index ----
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

# ---- STEP 5: Chat Loop with Retrieval ----
print("Company Chatbot Ready! Type 'quit' to exit.\n")

chat_history_ids = None
while True:
    query = input("You: ")
    if query.lower() in ["quit", "exit"]:
        print("Bot: Goodbye! 👋")
        break
    
    # Encode query
    query_emb = embedder.encode([query])
    
    # Retrieve most relevant doc
    D, I = index.search(np.array(query_emb), k=1)
    retrieved_doc = documents[I[0][0]]
    
    # Combine query + retrieved context
    prompt = f"Context: {retrieved_doc}\n\nQuestion: {query}\nAnswer:"
    
    # Encode for LLM
    input_ids = tokenizer.encode(prompt + tokenizer.eos_token, return_tensors="pt")
    bot_input_ids = torch.cat([chat_history_ids, input_ids], dim=-1) if chat_history_ids is not None else input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=500,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7
    )

    bot_reply = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    print("Bot:", bot_reply)


NameError: name 'LRScheduler' is not defined

In [13]:
pip install transformers accelerate sentence-transformers faiss-cpu

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/18.2 MB 8.4 MB/s eta 0:00:02
   --------- ------------------------------ 4.2/18.2 MB 9.7 MB/s eta 0:00:02
   ------------- -------------------------- 6.3/18.2 MB 9.9 MB/s eta 0:00:02
   ------------------- -------------------- 8.7/18.2 MB 10.1 MB/s eta 0:00:01
   ------------------------ --------------- 11.3/18.2 MB 10.5 MB/s eta 0:00:01
   ------------------------------ --------- 13.6/18.2 MB 10.7 MB/s eta 0:00:01
   ---------------------------------- ----- 15.5/18.2 MB 10.4 MB/s eta 0:00:01
   -------------------------------------- - 17.3/18.2 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 18.2/18.2 MB 9.7 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [faiss-cpu]
   -------------------


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
